In [2]:
# Add project root (the folder that contains "src/") to sys.path
import sys
import time
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root()  # looks for a "src" sibling/parent of the notebook


[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation


In [3]:
import os, json, time, math, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- your data utils ---
from src.data.prepare_dataset import (
    build_pairs_for_split,
    build_all_train_pairs,
    assert_dataset_layout,
    sanity_check_sample_alignment,
)
from src.data.dataloader import make_loaders

# --- your model pieces ---
from src.models.dpcn.dpcn_vat_exp1 import DPCN          # your DPCN (with threshold modes)
from src.models.blocks.cbam import CBAM               # your CBAM
from src.models.unet import UNet               # UNet that accepts in_channels (the variant we discussed)
from src.models.ablations.dpcn_cbam_unet import DPCN_CBAM_UNet  # the glue class we made earlier

# --- your evaluation & visualization ---
from src.evaluation.evaluate import evaluate_and_print
from src.evaluation.visualization import visualize_samples

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = Path("./runs/dpcn_cbam_unet")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SEED = 1337
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True


has op: True


In [4]:
DRIVE_ROOT = "../data/raw/DRIVE"
assert_dataset_layout(DRIVE_ROOT, split="training", label_folder="1st_manual")
train_pairs = build_pairs_for_split(DRIVE_ROOT, split="training", label_folder="1st_manual")
val_pairs   = build_pairs_for_split(DRIVE_ROOT, split="test",     label_folder="1st_manual")

print("sample alignment:", sanity_check_sample_alignment(train_pairs, k=3))

sample alignment: [('21_training.png', '21_manual1.png'), ('22_training.png', '22_manual1.png'), ('23_training.png', '23_manual1.png')]


In [5]:
RAW_ROOT = "../data/raw"
train_pairs = build_all_train_pairs(raw_root=RAW_ROOT, datasets=("DRIVE","CHASEDB1","STARE"), label_folder="1st_manual")
# keep DRIVE test as val, or build your own val_pairs:
val_pairs   = build_pairs_for_split(f"{RAW_ROOT}/DRIVE", split="test", label_folder="1st_manual")

print("train sample alignment:", sanity_check_sample_alignment(train_pairs, k=3))
print("val   sample alignment:", sanity_check_sample_alignment(val_pairs,   k=3))

train sample alignment: [('21_training.png', '21_manual1.png'), ('22_training.png', '22_manual1.png'), ('23_training.png', '23_manual1.png')]
val   sample alignment: [('01_test.png', '01_manual1.png'), ('02_test.png', '02_manual1.png'), ('03_test.png', '03_manual1.png')]


In [6]:
# You can pass your Albumentations augs via augs_train / augs_val if you have them.
train_loader, val_loader = make_loaders(
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    image_size=512,
    batch_size=4,
    num_workers=0, # to change to  4
    seed=SEED,
    strict_fov=True,
    augs_train=None,
    augs_val=None,
)

In [7]:
# DPCN core
dpcn = DPCN(
    in_ch=1,
    channels=64,           
    iters=5,             
    beta_init=0.3,
    aE=0.8,
    V_E=0.3,
    threshold_mode="paper_mod",   # start stable; later you can switch to "vat"/"scaled_vat"
    clamp_each_iter=False
)

# Full pipeline: DPCN -> concat(T*C) -> CBAM -> UNet(in_channels=T*C)
model_core = DPCN_CBAM_UNet(
    dpcn=dpcn,
    reduction_ratio=8,
    use_spatial_cbam=True
).to(DEVICE)

In [8]:
class LogitsOnly(nn.Module):
    """Wraps a (logits, ys, feats) model to expose logits-only forward(x[,fov])."""
    def __init__(self, core):
        super().__init__()
        self.core = core
    def forward(self, x):
        logits, _, _ = self.core(x)  # NOTE: no fov in eval/viz wrapper
        return logits

model_for_eval = LogitsOnly(model_core).to(DEVICE)

In [ ]:
# loss here

In [ ]:
def train_one_epoch(model_core, loader, optimizer, scaler):
    model_core.train()
    total = 0.0
    for batch in loader:
        x   = batch["image"].to(DEVICE)   # [B,1,H,W]
        y   = batch["mask"].to(DEVICE)    # [B,1,H,W]
        fov = batch["fov"].to(DEVICE)     # [B,1,H,W]

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, ys, feats = model_core(x, fov=fov)   # <- use FOV during training
            loss = loss_fn(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model_core.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * x.size(0)

    return total / len(loader.dataset)

@torch.no_grad()
def validate_loss_only(model_core, loader):
    model_core.eval()
    total = 0.0
    for batch in loader:
        x = batch["image"].to(DEVICE)
        y = batch["mask"].to(DEVICE)
        logits, _, _ = model_core(x)      # no FOV path in val loss; inputs are FOV-clamped already
        total += loss_fn(logits, y).item() * x.size(0)
    return total / len(loader.dataset)


In [ ]:
#modeltaining

In [ ]:
# load best
model_core.load_state_dict(torch.load(SAVE_DIR / "best.pth", map_location=DEVICE))
model_for_eval = LogitsOnly(model_core).to(DEVICE).eval()

# run your evaluation utility
evaluate_and_print(model_for_eval, test_dataloader=val_loader, device=DEVICE, threshold=0.5, compute_auc=True)

In [ ]:
visualize_samples(
    model=model_for_eval,
    dataloader=val_loader,
    n_rows=6,
    device=DEVICE,
    threshold=0.5,
    clamp_pred_with_fov=True,
    figsize_per_row=(12, 3),
)